In [ ]:
# ── All imports (run this cell first) ────────────────────────────────────────
import gc
import json
import os
import random
import re
import shutil
from pathlib import Path

import av
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
from huggingface_hub import snapshot_download
from natsort import natsorted
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from tqdm import tqdm
from transformers import (
    Qwen3OmniMoeProcessor,
    Qwen3OmniMoeThinkerForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
os.environ.pop("SOCKS_PROXY", None)
os.environ.pop("socks_proxy", None)
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)

import transformers
print(f"transformers version : {transformers.__version__}")
print(f"torch version        : {torch.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM                 : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Environment Verification

Findings from inspecting the installed `transformers` package (version 5.5.0):

| Item | Qwen3-Omni-7B value |
|------|---------------------|
| **Thinker-only class** | `Qwen3OmniMoeThinkerForConditionalGeneration` |
| **Processor class** | `Qwen3OmniMoeProcessor` |
| **Audio token ID** | `model.config.audio_token_id` → 151646 (direct field, **no** attribute_map alias) |
| **Video token ID** | `model.config.video_token_id` → 151656 (direct field) |
| **Layer count attr** | `model.config.text_config.num_hidden_layers` (default config says 28; real 7B model will differ) |
| **`attn_implementation="sdpa"`** | ✅ supported (`_supports_sdpa = True` on text layers) |
| **`output_hidden_states`** | passed via `**kwargs` → `self.model(...)` |
| **`qwen-omni-utils`** | NOT installed |
| **`qwen-vl-utils`** | version 0.0.8 — `process_vision_info` handles video frames |

**Qwen3-Omni architecture note:** Qwen3-Omni uses a **Mixture-of-Experts** (MoE) text backbone
(`Qwen3OmniMoe`), which is why the class names carry the `Moe` suffix.
This is distinct from Qwen2.5-Omni (dense transformer).
Class names, config attribute names, and processor attributes differ between the two model families.

**Key difference from Qwen2.5-Omni:**
- Qwen2.5: `audio_token_id` mapped via `attribute_map` to `audio_token_index`
- Qwen3: `audio_token_id` is a **direct** field on the config (no alias needed)

**Token strings** (verified from processor source inspection):
- `processor.audio_token` = `"<|AUDIO|>"` → ID 151646
- `processor.video_token` = `"<|VIDEO|>"` → ID 151656

**Target GPU:** NVIDIA RTX 4090 (24 GB VRAM).  
**MODEL_PATH must be updated** to point to the lab HDD copy before running.

In [ ]:
# ── Download Qwen3-Omni-7B to lab HDD (skip if already present) ───────────────
# Run this cell only when the HDD is attached and weights need to be downloaded.

EXTERNAL_HF_CACHE = "/path/to/lab/hdd/hf_models/hub"  # ← UPDATE before running
QWEN_MODEL_ID     = "Qwen/Qwen3-Omni-7B"

def _weights_present(ext_model_dir: Path) -> bool:
    snap_dir = ext_model_dir / "snapshots"
    if not snap_dir.exists():
        return False
    snaps = [s for s in snap_dir.iterdir() if s.is_dir()]
    if not snaps:
        return False
    snap = sorted(snaps)[-1]
    for idx_name in ("model.safetensors.index.json", "pytorch_model.bin.index.json"):
        idx_path = snap / idx_name
        if not idx_path.exists():
            continue
        idx = json.loads(idx_path.read_text())
        expected = set(idx["weight_map"].values())
        missing = [s for s in expected if not (snap / s).exists() or (snap / s).stat().st_size == 0]
        if missing:
            print(f"  Incomplete: {len(missing)}/{len(expected)} shards missing")
            return False
        print(f"  All {len(expected)} shards present in {snap.name[:12]}...")
        return True
    return any(f.suffix == ".safetensors" and "index" not in f.name for f in snap.iterdir())

ext_dir    = Path(EXTERNAL_HF_CACHE) / "models--Qwen--Qwen3-Omni-7B"
sym_target = Path.home() / ".cache" / "huggingface" / "hub" / "models--Qwen--Qwen3-Omni-7B"

if not _weights_present(ext_dir):
    print("Downloading Qwen3-Omni-7B ...")
    snapshot_download(QWEN_MODEL_ID, cache_dir=EXTERNAL_HF_CACHE,
                      ignore_patterns=["*.msgpack", "flax_model*"])
else:
    print(f"Weights already present: {ext_dir}")

if sym_target.exists() and not sym_target.is_symlink():
    shutil.rmtree(sym_target)
if not sym_target.exists():
    sym_target.symlink_to(ext_dir)
    print(f"Symlink created: {sym_target} -> {ext_dir}")
else:
    print(f"Symlink OK: {sym_target}")

In [ ]:
# ── SECTION 0 — CONFIG ────────────────────────────────────────────────────────
# All paths and parameters live here. Nothing is hardcoded elsewhere.

# Model: Qwen3-Omni-7B  (NOT Qwen2.5 — this is the Qwen3 generation, MoE backbone)
# !! UPDATE THIS PATH to the local HDD copy before running on the lab machine !!
MODEL_PATH = "/path/to/lab/hdd/Qwen3-Omni-7B"

DATA_BASE       = Path("/home/amin/Research/Representation/Movie/data/segmented_stimulus/filtered")
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")

DEVICE = "cuda"
DTYPE  = torch.bfloat16

# BIN_SEC: window duration in seconds; SKIP_SEC: stride (= BIN_SEC → no overlap)
BIN_SEC  = 5.0
SKIP_SEC = 5.0

# Each tuple: (bin_sec, skip_sec, segments_root)
SEGMENT_DURATIONS = [
    (5.0,  5.0,  DATA_BASE),
    (10.0, 10.0, DATA_BASE),
]
MODEL_TAG = "omni7b"   # used in all output filenames and directory names

# Audio sampling rate expected by WhisperFeatureExtractor inside this processor.
AUDIO_SR = 16000
# ─────────────────────────────────────────────────────────────────────────────
print(f"MODEL_TAG       : {MODEL_TAG}")
print(f"EMBEDDINGS_BASE : {EMBEDDINGS_BASE}")
EMBEDDINGS_BASE.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── SECTION 1 — MODEL LOADING ────────────────────────────────────────────────
# Load ONLY the Thinker component (MoE text backbone + audio encoder + vision encoder).
# The Talker (speech synthesis) is NOT loaded.
# Qwen3OmniMoeThinkerForConditionalGeneration is the Thinker-only class;
# it has no Talker attribute.
#
# NOTE: Qwen3-Omni-7B uses a Mixture-of-Experts (MoE) language model.
# The class names carry the 'Moe' suffix to distinguish from Qwen2.5-Omni's dense backbone.
#
# device_map={"": DEVICE} forces ALL sub-modules (audio encoder, vision encoder,
# LLM decoder) onto the same physical device, preventing cross-device tensor
# mismatches that can occur with device_map="auto" when a module is silently
# placed on CPU or a different GPU.
#
# MODEL_PATH must be updated to the local HDD path before running on the lab machine.

assert hasattr(transformers, "Qwen3OmniMoeThinkerForConditionalGeneration"), (
    "Qwen3OmniMoeThinkerForConditionalGeneration not found in this transformers version."
    " Need >= 5.x. Installed: " + transformers.__version__
)
assert hasattr(transformers, "Qwen3OmniMoeProcessor"), (
    "Qwen3OmniMoeProcessor not found in this transformers version."
)

torch.cuda.empty_cache()
gc.collect()

print("Loading Qwen3-Omni-7B Thinker (MoE) ...")
processor = Qwen3OmniMoeProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
)

model = Qwen3OmniMoeThinkerForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    device_map={"": DEVICE},   # force all modules to one device; avoids cross-device mismatch
    torch_dtype=DTYPE,
    attn_implementation="sdpa",
    local_files_only=True,
)
model.eval()

# Store the resolved device so all subsequent .to() calls are consistent.
_MODEL_DEVICE = next(model.parameters()).device
print(f"Thinker loaded on {_MODEL_DEVICE}.")

# ── Read layer count from config (never hardcode) ────────────────────────────
N_LAYERS = model.config.text_config.num_hidden_layers
# Quarter-depth probes + early layers (1, 2, 4) to capture low-level features.
# sorted(set(...)) deduplicates in case early indices overlap with quarter points.
QUARTER_LAYERS = sorted(set([
    1, 2, 4,
    N_LAYERS // 4,
    N_LAYERS // 2,
    3 * N_LAYERS // 4,
    N_LAYERS - 1,
]))
print(f"N_LAYERS       : {N_LAYERS}")
print(f"QUARTER_LAYERS : {QUARTER_LAYERS}")

# ── Verify token IDs from config (read dynamically, never hardcode) ───────────
# For Qwen3-Omni: audio_token_id and video_token_id are DIRECT fields on the config
# (unlike Qwen2.5-Omni where they are aliases via attribute_map).
AUDIO_TOKEN_ID = model.config.audio_token_id
VIDEO_TOKEN_ID = model.config.video_token_id
assert AUDIO_TOKEN_ID is not None, "model.config.audio_token_id is None — check model config"
assert VIDEO_TOKEN_ID is not None, "model.config.video_token_id is None — check model config"
print(f"audio_token_id : {AUDIO_TOKEN_ID}")
print(f"video_token_id : {VIDEO_TOKEN_ID}")

# Cross-check against processor's tokenizer to catch any mismatch.
_proc_audio_id = processor.tokenizer.convert_tokens_to_ids(processor.audio_token)
_proc_video_id = processor.tokenizer.convert_tokens_to_ids(processor.video_token)
assert _proc_audio_id == AUDIO_TOKEN_ID, (
    f"Audio token ID mismatch: config={AUDIO_TOKEN_ID}, processor={_proc_audio_id}"
)
assert _proc_video_id == VIDEO_TOKEN_ID, (
    f"Video token ID mismatch: config={VIDEO_TOKEN_ID}, processor={_proc_video_id}"
)
print("Token ID cross-check passed.")

# ── GPU memory after loading ──────────────────────────────────────────────────
torch.cuda.synchronize()
mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory allocated after loading: {mem_gb:.2f} GB")

In [ ]:
# ── SECTION 2 — TOKEN MASK UTILITIES ─────────────────────────────────────────
# Qwen3-Omni (omni7b): AUDIO_TOKEN_ID / VIDEO_TOKEN_ID are direct config fields
# (not aliased via attribute_map, unlike Qwen2.5-Omni).
# Both are cross-checked against processor.tokenizer in Section 1.

_masks_verified = False


def get_audio_token_mask(ids: torch.Tensor) -> torch.Tensor:
    """Bool mask (seq_len,) True at audio token positions."""
    return ids == AUDIO_TOKEN_ID


def get_video_token_mask(ids: torch.Tensor) -> torch.Tensor:
    """Bool mask (seq_len,) True at video token positions."""
    return ids == VIDEO_TOKEN_ID


def get_av_token_mask(ids: torch.Tensor) -> torch.Tensor:
    """Bool mask (seq_len,) True at audio OR video token positions."""
    return get_audio_token_mask(ids) | get_video_token_mask(ids)


def get_all_masks(input_ids: torch.Tensor) -> tuple:
    """Return (av_mask, a_mask, v_mask) as CPU bool tensors of shape (seq_len,).

    First call prints a verification report and asserts independently:
    - at least one audio token found
    - at least one video token found
    - |av_mask| == |a_mask| + |v_mask|  (IDs are disjoint, no double-counting)

    input_ids: (1, seq_len) or (seq_len,).
    """
    global _masks_verified
    ids     = input_ids.squeeze(0)
    a_mask  = get_audio_token_mask(ids)
    v_mask  = get_video_token_mask(ids)
    av_mask = a_mask | v_mask
    if not _masks_verified:
        n_a  = int(a_mask.sum())
        n_v  = int(v_mask.sum())
        n_av = int(av_mask.sum())
        n_t  = int((~av_mask).sum())
        print(f"[Mask verify]  audio_id={AUDIO_TOKEN_ID}  video_id={VIDEO_TOKEN_ID}")
        print(f"[Mask verify]  audio={n_a}  video={n_v}  av={n_av}  text={n_t}  total={len(ids)}")
        assert n_a  > 0, f"No audio tokens — AUDIO_TOKEN_ID={AUDIO_TOKEN_ID} not in sequence"
        assert n_v  > 0, f"No video tokens — VIDEO_TOKEN_ID={VIDEO_TOKEN_ID} not in sequence"
        assert n_av == n_a + n_v, "AV count != audio+video — token ID overlap detected"
        print("[Mask verify]  all assertions passed.")
        _masks_verified = True
    return av_mask.cpu(), a_mask.cpu(), v_mask.cpu()

In [ ]:
# ── SECTION 3 — DYNAMIC BATCH SIZE ───────────────────────────────────────────

# Default system prompt required to suppress Qwen's "System prompt modified" warning.
_QWEN_SYSTEM_PROMPT = (
    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
)


def _load_audio(path: Path, target_sr: int) -> np.ndarray:
    """Decode audio from an mp4 or wav file via PyAV."""
    with av.open(str(path)) as container:
        stream = next((s for s in container.streams if s.type == "audio"), None)
        if stream is None:
            return np.zeros(target_sr, dtype=np.float32)
        native_sr = stream.sample_rate
        chunks = []
        for frame in container.decode(stream):
            arr = frame.to_ndarray().astype(np.float32)
            if arr.ndim > 1:
                arr = arr.mean(axis=0)
            chunks.append(arr)
    if not chunks:
        return np.zeros(target_sr, dtype=np.float32)
    audio = np.concatenate(chunks)
    if native_sr != target_sr:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio), native_sr, target_sr
        ).numpy()
    return audio


def _build_inputs_for_video(video_path: Path) -> tuple:
    """Build processor inputs for a single video+audio file."""
    msgs = [
        # content must be a list-of-dicts, not a plain string —
        # apply_chat_template does conversation[0]["content"][0]["text"]
        {"role": "system", "content": [{"type": "text", "text": _QWEN_SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "video", "video": str(video_path)},
            {"type": "audio", "audio": str(video_path)},
            {"type": "text",  "text":  "Describe what you see and hear."},
        ]},
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    _, frames = process_vision_info(msgs)
    audio_array = _load_audio(video_path, AUDIO_SR)
    return text, frames, audio_array


def find_safe_batch_size(
    sample_video_path: Path,
    max_batch: int = 8,
) -> int:
    """Probe batch sizes 1..max_batch; return (largest safe - 1), minimum 1."""
    text_single, frames_single, audio_single = _build_inputs_for_video(sample_video_path)
    last_ok = 0
    for bs in range(1, max_batch + 1):
        torch.cuda.empty_cache()
        try:
            texts  = [text_single] * bs
            frames = frames_single * bs
            audios = np.stack([audio_single] * bs, axis=0) if bs > 1 else audio_single
            inputs = processor(
                text=texts,
                videos=frames,
                audio=audios,
                sampling_rate=AUDIO_SR,
                return_tensors="pt",
            ).to(_MODEL_DEVICE)
            with torch.inference_mode():
                _ = model(**inputs, output_hidden_states=False)
            last_ok = bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"  OOM at batch_size={bs}")
            break
        finally:
            torch.cuda.empty_cache()
    safe = max(1, last_ok - 1)
    print(f"Safe batch size: {safe}  (last successful: {last_ok})")
    return safe

In [ ]:
# ── SECTION 4 — SINGLE SEGMENT EXTRACTION (hook-based, memory-efficient) ──────
# output_hidden_states=True accumulates all N_LAYERS tensors in GPU RAM at once.
# Even on a 4090 (24 GB), a 7B MoE model leaves limited headroom for long sequences.
# Forward hooks capture only the quarter layers and move each to CPU immediately.
#
# Whisper padding: the audio feature extractor always pads to 30s → 1500 encoder
# tokens, but short clips fill only the leading tokens with real audio. We compute
# n_valid_audio from the raw sample count and pool only over those leading positions.
#   n_mel ≈ ceil(n_samples / 160)     (Whisper hop = 160 samples at 16 kHz)
#   n_valid_audio = ceil(n_mel / 2)   (Conv1d stride-2 in encoder)
#
# Returned masks always use the trimmed audio count so that _a and _av embeddings
# are not diluted by ~5x silence for 5-second segments.

_audio_trim_logged = False


def _find_decoder_layers(n_expected: int) -> torch.nn.ModuleList:
    """Find the LLM decoder-layer ModuleList (works for dense and MoE Qwen3).

    Qwen3 decoder layers (dense or MoE) always have both self_attn and
    input_layernorm. Whisper encoder layers use self_attn_layer_norm instead,
    so the input_layernorm check reliably excludes them.
    """
    for _, mod in model.named_modules():
        if (isinstance(mod, torch.nn.ModuleList)
                and len(mod) == n_expected
                and hasattr(mod[0], 'self_attn')
                and hasattr(mod[0], 'input_layernorm')):
            return mod
    raise RuntimeError(f"Could not find Qwen3 decoder ModuleList of length {n_expected}")

_decoder_layers = _find_decoder_layers(N_LAYERS)
print(f"Decoder layers found: {len(_decoder_layers)}")


def extract_av_embeddings(video_path: Path, target_layers: list | None = None):
    """Extract hidden states using forward hooks (memory-efficient).

    Audio mask is trimmed to the first n_valid_audio positions to exclude
    Whisper padding tokens that carry no real audio signal.

    Parameters
    ----------
    target_layers : list of int  (default: QUARTER_LAYERS)
        Layer indices to capture (0-indexed into the decoder stack).

    Returns
    -------
    hidden_states : dict  {layer_idx → tensor(1, seq_len, D) on CPU}
    av_mask       : bool tensor (seq_len,) on CPU — trimmed audio + video positions
    a_mask        : bool tensor (seq_len,) on CPU — trimmed audio positions only
    v_mask        : bool tensor (seq_len,) on CPU — all video positions
    """
    global _audio_trim_logged
    if target_layers is None:
        target_layers = QUARTER_LAYERS

    text, frames, audio_array = _build_inputs_for_video(video_path)

    # Compute how many leading audio tokens carry real signal (before Whisper padding)
    n_samples = len(audio_array)
    n_mel = int(np.ceil(n_samples / 160))
    n_valid_audio = min(int(np.ceil(n_mel / 2)), 1500)

    inputs = processor(
        text=[text],
        videos=frames,
        audio=audio_array,
        sampling_rate=AUDIO_SR,
        return_tensors="pt",
    ).to(_MODEL_DEVICE)

    captured: dict[int, torch.Tensor] = {}
    handles  = []

    def _make_hook(idx: int):
        def _hook(module, inp, out):
            hs = out[0] if isinstance(out, (tuple, list)) else out
            captured[idx] = hs.detach().cpu()
        return _hook

    for i in target_layers:
        handles.append(_decoder_layers[i].register_forward_hook(_make_hook(i)))

    try:
        with torch.inference_mode():
            model(**inputs, output_hidden_states=False)
    finally:
        for h in handles:
            h.remove()

    # Get full masks (verification fires on first call), then trim audio positions
    _, a_mask_full, v_mask = get_all_masks(inputs["input_ids"])
    del inputs
    torch.cuda.empty_cache()

    a_positions = a_mask_full.nonzero(as_tuple=True)[0]
    n_clip = min(n_valid_audio, len(a_positions))
    a_mask = torch.zeros_like(a_mask_full)
    a_mask[a_positions[:n_clip]] = True
    av_mask = a_mask | v_mask

    if not _audio_trim_logged:
        print(f"[Audio trim]  valid={n_clip}/{len(a_positions)} audio tokens "
              f"(n_samples={n_samples}, n_mel={n_mel}, n_valid_audio={n_valid_audio})")
        _audio_trim_logged = True

    return captured, av_mask, a_mask, v_mask

In [ ]:
# ── SECTION 5 — MEAN POOL EMBEDDING ──────────────────────────────────────────

def get_mean_pool_embedding(
    hidden_states: dict,
    mask: torch.Tensor,
    layer_idx: int,
) -> np.ndarray:
    """Mean pool hidden states at positions where mask is True.

    Parameters
    ----------
    hidden_states : dict {layer_idx → tensor(1, seq_len, D)} — all on CPU
    mask          : bool tensor (seq_len,) on CPU — av_mask, a_mask, or v_mask
    layer_idx     : which layer (must be a key in hidden_states)

    Returns
    -------
    np.ndarray of shape (hidden_dim,). Returns zeros if mask is all-False.
    """
    if layer_idx not in hidden_states:
        available = sorted(hidden_states.keys())
        raise KeyError(f"layer_idx={layer_idx} not in captured layers {available}")
    hs = hidden_states[layer_idx]  # (1, seq_len, D) on CPU
    if not mask.any():
        return np.zeros(hs.shape[-1], dtype=np.float32)
    return hs[0, mask, :].mean(dim=0).float().numpy()

In [ ]:
# ── SECTION 6 — EMPIRICAL LAYER SELECTION ────────────────────────────────────
# Sample 50 segments from the primary bin duration, extract QUARTER_LAYERS
# embeddings, compute RDMs, and compare Spearman r against the last-layer RDM.

_dur_int  = int(BIN_SEC)
_skip_int = int(SKIP_SEC)
_chunk_suffix = (f"_av_chunks_{_dur_int}s" if _skip_int == _dur_int
                 else f"_av_chunks_{_dur_int}s_skip{_skip_int}s")
all_segs = natsorted(list(DATA_BASE.rglob(f"*{_chunk_suffix}/*.mp4")), key=lambda p: p.name)
assert len(all_segs) > 0, f"No {BIN_SEC}s AV .mp4 files found under {DATA_BASE}"
sample_size = min(50, len(all_segs))
random.seed(42)
sample_paths = random.sample(all_segs, sample_size)
print(f"Sampling {sample_size} / {len(all_segs)} {_dur_int}s segments for layer selection.")
print(f"Probing layers: {QUARTER_LAYERS}")

layer_embeddings = {idx: [] for idx in QUARTER_LAYERS}
failed_sample = []

for vp in tqdm(sample_paths, desc="Layer selection sample"):
    try:
        hs, av_mask, a_mask, v_mask = extract_av_embeddings(vp)
        for idx in QUARTER_LAYERS:
            e_a = get_mean_pool_embedding(hs, a_mask, idx)
            e_v = get_mean_pool_embedding(hs, v_mask, idx)
            layer_embeddings[idx].append((e_a + e_v) / 2.0)  # equal modality weight
        del hs
        torch.cuda.empty_cache()
    except Exception as e:
        failed_sample.append((vp.name, repr(e)))

if failed_sample:
    print(f"\n{len(failed_sample)} segments failed during sampling:")
    for name, err in failed_sample:
        print(f"  {name}: {err}")

# ── RDMs and Spearman r ───────────────────────────────────────────────────────
rdms = {}
for idx in QUARTER_LAYERS:
    embs_list = layer_embeddings[idx]
    if len(embs_list) == 0:
        print(f"Layer {idx}: no embeddings collected — skipping RDM.")
        continue
    embs = np.array(embs_list)  # (N_sample, D)
    rdms[idx] = squareform(pdist(embs, metric="cosine"))

if N_LAYERS - 1 not in rdms:
    print("Last-layer RDM unavailable — cannot compute Spearman r. Check failures above.")
else:
    last_layer_rdm = rdms[N_LAYERS - 1].flatten()
    print("\nSpearman r vs last-layer RDM:")
    for idx in QUARTER_LAYERS:
        if idx not in rdms:
            print(f"  Layer {idx:3d}: skipped (no embeddings)")
            continue
        r, _ = spearmanr(rdms[idx].flatten(), last_layer_rdm)
        label = " (last)" if idx == N_LAYERS - 1 else ""
        print(f"  Layer {idx:3d}: r = {r:.3f}{label}")

    # ── Dynamic RDM grid (adapts to any number of layers) ────────────────────
    valid_layers = [idx for idx in QUARTER_LAYERS if idx in rdms]
    n_show = len(valid_layers)
    ncols = min(4, n_show)
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    axes_flat = axes.flatten()
    for ax in axes_flat:
        ax.set_visible(False)
    for ax, idx in zip(axes_flat, valid_layers):
        ax.set_visible(True)
        im = ax.imshow(rdms[idx], vmin=0, vmax=1, cmap="hot_r")
        ax.set_title(f"Layer {idx}")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"{MODEL_TAG} — layer RDMs (N={sample_size} segments)")
    plt.tight_layout()
    plt.show()

# ── Save sample embeddings ────────────────────────────────────────────────────
for idx in QUARTER_LAYERS:
    embs_list = layer_embeddings[idx]
    if len(embs_list) == 0:
        continue
    arr = np.array(embs_list)
    layer_dir = EMBEDDINGS_BASE / f"{MODEL_TAG}_layer{idx}"
    layer_dir.mkdir(parents=True, exist_ok=True)
    out_path = layer_dir / f"{MODEL_TAG}_layer{idx}_sample_av_{_dur_int}s.npy"
    np.save(out_path, arr)
    print(f"Saved {out_path.name}  shape={arr.shape}")

These RDMs should be correlated against brain ROI RSA results from PE-AV analysis before committing to a layer for full extraction. The last layer is used as primary but may not be optimal.

In [ ]:
# ── SECTION 7 — FULL EXTRACTION LOOP ─────────────────────────────────────────
# Loops over SEGMENT_DURATIONS and extracts quarter-layer mean-pool embeddings
# for three modality masks per segment:
#   _av : equal-weight average of audio and video mean pools
#   _a  : audio tokens only
#   _v  : video tokens only
#
# _av = (mean_audio_pool + mean_video_pool) / 2 gives equal weight to each
# modality regardless of token count (video tokens greatly outnumber audio).
#
# Output structure:
#   EMBEDDINGS_BASE/{MODEL_TAG}_layer{idx}/{out_tag}/{MODEL_TAG}_layer{idx}_{suffix}.npy
#
# Skip logic: if all 3 × len(QUARTER_LAYERS) outputs exist for a duration, skip it.
# GPU clear: between durations to free VRAM before processing longer segments.

for bin_sec, skip_sec, segments_dir in SEGMENT_DURATIONS:
    dur_int  = int(bin_sec)
    skip_int = int(skip_sec)
    chunk_suffix = (f"_av_chunks_{dur_int}s" if skip_int == dur_int
                    else f"_av_chunks_{dur_int}s_skip{skip_int}s")
    mp4_files = natsorted(
        list(segments_dir.rglob(f"*{chunk_suffix}/*.mp4")),
        key=lambda p: p.name,
    )
    out_tag = f"bin{dur_int}s_skip{skip_int}s"
    if len(mp4_files) == 0:
        print(f"[{out_tag}] No AV .mp4 files found — skipping.")
        continue

    # ── Skip if all outputs already exist ────────────────────────────────────
    expected = [
        EMBEDDINGS_BASE / f"{MODEL_TAG}_layer{idx}" / out_tag / f"{MODEL_TAG}_layer{idx}_{sfx}.npy"
        for idx in QUARTER_LAYERS
        for sfx in ("av", "a", "v")
    ]
    if all(p.exists() for p in expected):
        print(f"[{out_tag}] All {len(expected)} outputs exist — skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"{MODEL_TAG} — {out_tag}  ({len(mp4_files)} segments)")
    print(f"{'='*60}")

    _batch_size = find_safe_batch_size(mp4_files[0])
    print(f"Using batch_size={_batch_size} (one-by-one with safe probe).")

    embs_av = {idx: [] for idx in QUARTER_LAYERS}
    embs_a  = {idx: [] for idx in QUARTER_LAYERS}
    embs_v  = {idx: [] for idx in QUARTER_LAYERS}
    failed  = []

    for vp in tqdm(mp4_files, desc=f"{MODEL_TAG} {out_tag}"):
        try:
            hs, av_mask, a_mask, v_mask = extract_av_embeddings(vp)
            for idx in QUARTER_LAYERS:
                e_a = get_mean_pool_embedding(hs, a_mask, idx)
                e_v = get_mean_pool_embedding(hs, v_mask, idx)
                embs_av[idx].append((e_a + e_v) / 2.0)  # equal modality weight
                embs_a[idx].append(e_a)
                embs_v[idx].append(e_v)
            del hs
            torch.cuda.empty_cache()
        except Exception as e:
            err_str = repr(e)
            failed.append((vp.name, err_str))
            print(f"\n  FAILED {vp.name}: {err_str}")

    # ── Save per-layer, per-modality ──────────────────────────────────────────
    for idx in QUARTER_LAYERS:
        layer_dir = EMBEDDINGS_BASE / f"{MODEL_TAG}_layer{idx}" / out_tag
        layer_dir.mkdir(parents=True, exist_ok=True)
        for sfx, emb_list in [("av", embs_av[idx]), ("a", embs_a[idx]), ("v", embs_v[idx])]:
            arr = np.array(emb_list)  # (N, D)
            out_path = layer_dir / f"{MODEL_TAG}_layer{idx}_{sfx}.npy"
            np.save(out_path, arr)
            print(f"  saved {out_path.relative_to(EMBEDDINGS_BASE)}  shape={arr.shape}")

    if failed:
        print(f"\n  Failed segments ({len(failed)}):")
        for name, err in failed:
            print(f"    {name}: {err}")
    else:
        print(f"  All {len(mp4_files)} segments extracted successfully.")

    # ── Clear GPU and CPU buffers before next duration ────────────────────────
    del embs_av, embs_a, embs_v
    gc.collect()
    torch.cuda.empty_cache()

print("\nFull extraction complete.")

In [ ]:
# ── SECTION 8 — SANITY CHECKS ─────────────────────────────────────────────────
# Load the saved last-layer av / a / v embeddings for the primary duration.

_out_tag    = f"bin{int(BIN_SEC)}s_skip{int(SKIP_SEC)}s"
_last_layer = N_LAYERS - 1
_layer_dir  = EMBEDDINGS_BASE / f"{MODEL_TAG}_layer{_last_layer}" / _out_tag

for sfx in ("av", "a", "v"):
    fpath = _layer_dir / f"{MODEL_TAG}_layer{_last_layer}_{sfx}.npy"
    print(f"\nLoading: {fpath}")
    embs = np.load(fpath)
    print(f"  Shape : {embs.shape}")
    print(f"  Mean  : {embs.mean():.4f}  Std: {embs.std():.4f}  "
          f"Min: {embs.min():.4f}  Max: {embs.max():.4f}")
    if embs.std() < 0.01:
        print(f"  WARNING: {sfx} embeddings may be degenerate")

# RDM from first 100 segments of the av embeddings
av_path = _layer_dir / f"{MODEL_TAG}_layer{_last_layer}_av.npy"
embs_av = np.load(av_path)
n_check = min(100, len(embs_av))
rdm_check = squareform(pdist(embs_av[:n_check], metric="cosine"))
mean_off_diag = rdm_check[np.triu_indices_from(rdm_check, k=1)].mean()
print(f"\nMean off-diagonal cosine distance AV (N={n_check}): {mean_off_diag:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(rdm_check, vmin=0, vmax=1, cmap="hot_r")
ax.set_title(f"{MODEL_TAG} layer{_last_layer} — {_out_tag} AV RDM (N={n_check})")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print("\nSanity check complete.")